# Where PROJ finds `proj.db`, on the driver and in a UDF

**Purpose:** a light CRS call on a DBR 19 standard-access cluster raised `CRSError: ... Cannot
find proj.db` while the same call ran on dedicated access. This probe records, on the driver and
inside a Python UDF, where pyproj and rasterio look for their PROJ data, whether that directory
and its `proj.db` are visible and readable, and the `PROJ_*` / `GDAL_DATA` environment, before
and after GeoBrix is imported. It then calls one light function per family that resolves a CRS.

Run it on a DBR 19 job cluster in each access mode and compare:

```bash
bash scripts/commands/gbx-test-notebooks-serverless.sh \
  --notebook examples/tests/probe_proj_data_access.ipynb \
  --new-cluster --access-mode standard   # then --access-mode dedicated
```

| Row | Call | Expectation |
|---|---|---|
| raw | `pyproj.CRS.from_epsg`, `rasterio.crs.CRS.from_epsg` / `from_user_input` before GeoBrix is imported | resolve EPSG:4326 |
| gbx | `core.crs.resolve_crs` / `crs_to_canonical` after GeoBrix is imported | `EPSG:4326` |
| raster | `rst_srid(rst_transformcrs(tile, 'EPSG:4326'))` on a 4 x 4 EPSG:32630 tile | 4326 |
| vector | `gbx_st_transformcrs` of a point from EPSG:4326 to EPSG:27700 | easting about 530,000 |
| readers | `gtiff_gbx` tile SRID, `geojson_gbx` geometry SRID | 32630, 27700 |
| zarr | `gbx_zarr_sample` of two points on the ERA5 store | rows, no `ZarrUnsupported` |

The notebook ends with `dbutils.notebook.exit(<json>)`, which the runner prints.


In [ ]:
%pip install "geobrix[light_dbr19,zarr_dbr19] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.6.0-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
import json


def gbx_proj_probe(stage):
    """Where PROJ data is looked for, and whether a CRS resolves, before and after GeoBrix."""
    import hashlib
    import os
    import sys

    def env():
        keep = ("VIRTUAL_ENV", "PYTHONPATH", "HOME", "TMPDIR", "LD_LIBRARY_PATH")
        return {
            k: v
            for k, v in sorted(os.environ.items())
            if k.startswith(("PROJ", "GDAL", "GBX")) or k in keep
        }

    def look(path):
        if not path:
            return None
        db = os.path.join(path, "proj.db")
        r = {"path": path, "isdir": os.path.isdir(path), "proj_db": os.path.exists(db)}
        try:
            with open(db, "rb") as fh:
                r["read"] = len(fh.read(16))
        except Exception as e:  # noqa: BLE001
            r["read"] = f"{type(e).__name__}: {e}"[:200]
        return r

    def attempt(fn):
        try:
            return {"ok": fn()}
        except Exception as e:  # noqa: BLE001
            return {"error": f"{type(e).__name__}: {e}"[:300]}

    out = {"stage": stage, "pid": os.getpid(), "uid": os.getuid()}
    try:
        import pwd

        out["user"] = pwd.getpwuid(os.getuid()).pw_name
    except Exception as e:  # noqa: BLE001
        out["user"] = repr(e)
    out["executable"] = sys.executable
    out["env_before"] = env()
    out["usr_share_proj"] = look("/usr/share/proj")

    import pyproj
    import rasterio
    from pyproj.datadir import get_data_dir
    from rasterio._env import PROJDataFinder, get_proj_data_search_paths
    from rasterio.crs import CRS as RioCRS

    out["pyproj"] = {
        "version": pyproj.__version__,
        "file": pyproj.__file__,
        "proj": pyproj.proj_version_str,
        "data_dir": [look(d) for d in get_data_dir().split(os.pathsep)],
    }
    out["rasterio"] = {
        "version": rasterio.__version__,
        "file": rasterio.__file__,
        "gdal": rasterio.__gdal_version__,
        "proj": rasterio.__proj_version__,
        "wheel_proj_data": look(PROJDataFinder().search_wheel()),
        "search_paths": [look(p) for p in (get_proj_data_search_paths() or [])],
    }
    out["raw"] = {
        "pyproj.CRS.from_epsg(4326)": attempt(lambda: pyproj.CRS.from_epsg(4326).to_epsg()),
        "rasterio CRS.from_epsg(4326)": attempt(lambda: RioCRS.from_epsg(4326).to_epsg()),
        "rasterio CRS.from_user_input('EPSG:4326')": attempt(
            lambda: RioCRS.from_user_input("EPSG:4326").to_epsg()
        ),
    }

    import databricks.labs.gbx.pyrx  # noqa: F401  (configure_gdal_env runs at import)
    from databricks.labs.gbx.core.crs import crs_to_canonical, resolve_crs
    from databricks.labs.gbx.pyrx import _env as gbx_env

    with open(gbx_env.__file__, "rb") as fh:
        out["gbx_env_sha256"] = hashlib.sha256(fh.read()).hexdigest()[:16]
    out["env_after"] = env()
    out["rasterio_search_paths_after"] = [
        look(p) for p in (get_proj_data_search_paths() or [])
    ]
    out["gbx"] = {
        "resolve_crs('EPSG:4326')": attempt(lambda: resolve_crs("EPSG:4326").to_epsg()),
        "resolve_crs(4326)": attempt(lambda: resolve_crs(4326).to_epsg()),
        "crs_to_canonical(resolve_crs('EPSG:4326'))": attempt(
            lambda: crs_to_canonical(resolve_crs("EPSG:4326"))
        ),
        "resolve_crs(27700)": attempt(lambda: resolve_crs(27700).to_epsg()),
    }
    return json.dumps(out)


RESULTS = {"driver": json.loads(gbx_proj_probe("driver"))}
print(json.dumps(RESULTS["driver"], indent=1))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


@F.udf(T.StringType())
def _probe_udf(i):
    return gbx_proj_probe(f"udf-{i}")


_rows = spark.range(2).repartition(2).select(_probe_udf("id").alias("p")).collect()
RESULTS["udf"] = [json.loads(r["p"]) for r in _rows]
for _r in RESULTS["udf"]:
    print(json.dumps(_r, indent=1))

In [ ]:
import base64
import os
import uuid

import shapely
from pyspark.sql import functions as F

from databricks.labs.gbx import ds
from databricks.labs.gbx.pyrx import functions as prx
from databricks.labs.gbx.pyvx import functions as pvx

TIF = base64.b64decode("SUkqAAgAAAAPAAABAwABAAAABAAAAAEBAwABAAAABAAAAAIBAwABAAAACAAAAAMBAwABAAAAAQAAAAYBAwABAAAAAQAAABEBBAABAAAAaAEAABUBAwABAAAAAQAAABYBAwABAAAABAAAABcBBAABAAAAEAAAABwBAwABAAAAAQAAAFMBAwABAAAAAQAAAA6DDAADAAAAwgAAAIKEDAAGAAAA2gAAAK+HAwAgAAAACgEAALGHAgAeAAAASgEAAAAAAAAAAAAAAAAkQAAAAAAAACRAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAcFwlQQAAAABeyFVBAAAAAAAAAAABAAEAAAAHAAAEAAABAAEAAQQAAAEAAQACBLGHFgAAAAEIsYcHABYABggAAAEAjiMADAAAAQB2fwQMAAABACkjV0dTIDg0IC8gVVRNIHpvbmUgMzBOfFdHUyA4NHwAAAECAwQFBgcICQoLDA0ODw==")  # 4 x 4 uint8, EPSG:32630
SCRATCH = f"/Volumes/geospatial_docs/geobrix/sample-data/_probe/task61/{uuid.uuid4().hex[:8]}"
ZARR_STORE = (
    "/Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr-spike/read/era5/"
    "era5_v3_shard_zstd.zarr"
)
GEOJSON = json.dumps(
    {
        "type": "FeatureCollection",
        "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::27700"}},
        "features": [
            {
                "type": "Feature",
                "properties": {"name": "a"},
                "geometry": {"type": "Point", "coordinates": [530000.0, 180000.0]},
            }
        ],
    }
)

prx.register(spark)
pvx.register(spark)
ds.register.register(spark)
FN = {}


def row(name, fn):
    try:
        FN[name] = {"ok": fn()}
    except Exception as e:  # noqa: BLE001
        FN[name] = {"error": f"{type(e).__name__}: {e}"[:600]}
    print(name, "->", FN[name])


def raster_crs():
    df = spark.createDataFrame([(TIF,)], "content binary")
    tile = prx.rst_fromcontent(F.col("content"), F.lit("GTiff"))
    out = df.select(
        prx.rst_srid(tile).alias("src"),
        prx.rst_srid(prx.rst_transformcrs(tile, F.lit("EPSG:4326"))).alias("dst"),
    ).collect()[0]
    return {"src": out["src"], "dst": out["dst"]}


def vector_crs():
    r = spark.sql(
        "SELECT gbx_st_transformcrs('POINT (-0.1 51.5)', 'EPSG:27700', 'EPSG:4326') AS g"
    ).collect()[0]
    if r["g"] is None:
        return None
    p = shapely.from_wkb(bytes(r["g"]))
    return {"x": round(p.x, 1), "y": round(p.y, 1), "srid": int(shapely.get_srid(p))}


def reader_raster():
    os.makedirs(SCRATCH, exist_ok=True)
    with open(f"{SCRATCH}/tiny.tif", "wb") as fh:
        fh.write(TIF)
    df = spark.read.format("gtiff_gbx").load(f"{SCRATCH}/tiny.tif")
    return [r["s"] for r in df.select(prx.rst_srid("tile").alias("s")).collect()]


def reader_vector():
    os.makedirs(SCRATCH, exist_ok=True)
    with open(f"{SCRATCH}/tiny.geojson", "w") as fh:
        fh.write(GEOJSON)
    rows = spark.read.format("geojson_gbx").load(f"{SCRATCH}/tiny.geojson").collect()
    return [{k: v for k, v in r.asDict().items() if k.endswith("_srid")} for r in rows]


def zarr_sample():
    from databricks.labs.gbx.zarr import gbx_zarr_sample

    pts = spark.createDataFrame([("POINT (-0.1 51.5)",), ("POINT (10 45)",)], "geom string")
    out = gbx_zarr_sample(spark, pts, ZARR_STORE, variables="t2m")
    return {"rows": out.count(), "columns": out.columns}


row("raster: rst_transformcrs + rst_srid", raster_crs)
row("vector: gbx_st_transformcrs", vector_crs)
row("reader: gtiff_gbx rst_srid", reader_raster)
row("reader: geojson_gbx srid", reader_vector)
row("zarr: gbx_zarr_sample", zarr_sample)
RESULTS["functions"] = FN

In [ ]:
import shutil

shutil.rmtree(SCRATCH, ignore_errors=True)
dbutils.notebook.exit(json.dumps(RESULTS))  # surface the result to jobs.get_run_output